# RL Email Triage — Colab Training Notebook

Upload **only this `.ipynb` file** to Colab. Run cells top to bottom.

- Cell 1   : Setup paths
- **Cell 1.5 : Write all project source files to disk** ← makes notebook self-contained
- Cell 2   : Check GPU
- Cell 3   : Download Enron dataset
- Cell 4   : Verify imports
- Cell 5   : Train DQN (100,000 episodes)
- Cell 6   : Train Q-Learning (100,000 episodes)
- Cell 7   : Done


In [ ]:
# Cell 1: Project Path Setup
import os, sys

_cwd = os.path.abspath("")
if "email_response_timing" in _cwd:
    PROJECT_DIR = _cwd[: _cwd.find("email_response_timing") + len("email_response_timing(REL)")]
elif os.path.exists(r"C:\Users\Windows\Desktop\projects\email_response_timing(REL)"):
    PROJECT_DIR = r"C:\Users\Windows\Desktop\projects\email_response_timing(REL)"
else:
    PROJECT_DIR = _cwd  # /content on Colab

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.chdir(PROJECT_DIR)

ENRON_DIR = os.path.join(PROJECT_DIR, "enron_dataset")
MODELS_DIR = os.path.join(PROJECT_DIR, "models")
CKPT_DIR = os.path.join(MODELS_DIR, "checkpoints")
DQN_PATH = os.path.join(MODELS_DIR, "dqn.pkl")
QL_PATH = os.path.join(MODELS_DIR, "qlearning.pkl")

os.makedirs(ENRON_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print("PROJECT_DIR :", PROJECT_DIR)
print("MODELS_DIR  :", MODELS_DIR)
print("Folders confirmed OK")

In [ ]:
# Cell 1.5: Write all project source files to disk
# This makes the notebook fully self-contained — no ZIP or upload needed.
# Re-run safe: existing files are overwritten with the latest source.
import os

try:
    _root = PROJECT_DIR
except NameError:
    _root = os.path.abspath("")


def _write(rel_path, content):
    full = os.path.join(_root, rel_path)
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w", encoding="utf-8") as f:
        f.write(content)


# ── config.py ────────────────────────────────────────────────────────────────
_write(
    "config.py",
    """
class Config:
    STATE_SIZE  = 5
    ACTION_SIZE = 4
    MAX_STEPS   = 50
    SEED        = 42
    EPISODES     = 1_000_000
    LOG_EVERY    = 10_000
    CHECKPOINT_EVERY = 50_000
    EPISODES_LOCAL = 1_000
    DQN_ALPHA         = 0.0005
    DQN_GAMMA         = 0.95
    DQN_EPSILON       = 1.0
    DQN_EPSILON_MIN   = 0.01
    DQN_EPSILON_DECAY = 0.9999995
    DQN_BATCH_SIZE    = 128
    DQN_TARGET_UPDATE = 1_000
    DQN_BUFFER_CAP    = 100_000
    QL_ALPHA          = 0.1
    QL_GAMMA          = 0.95
    QL_EPSILON        = 1.0
    QL_EPSILON_MIN    = 0.01
    QL_EPSILON_DECAY  = 0.999993
    MODEL_DIR         = "models"
    DQN_MODEL_PATH    = "models/dqn.pkl"
    QL_MODEL_PATH     = "models/qlearning.pkl"
    FLASK_HOST  = "127.0.0.1"
    FLASK_PORT  = 5000
    FLASK_DEBUG = False
""",
)

# ── utils/ ───────────────────────────────────────────────────────────────────
_write("utils/__init__.py", "")
_write(
    "utils/logger.py",
    """
from datetime import datetime

class Logger:
    def __init__(self, log_every=100):
        self.log_every = log_every

    def episode(self, ep, reward, epsilon, q_size):
        if ep % self.log_every == 0:
            ts = datetime.now().strftime("%H:%M:%S")
            print(f"[{ts}] Episode {ep:>5} | Reward: {reward:>+7.1f} | "
                  f"Epsilon: {epsilon:.3f} | Buffer/States: {q_size}")

    def section(self, title):
        print("\\n" + "-" * 55)
        print(f"  {title}")
        print('-' * 55)

    def done(self, message):
        print(f"\\n[DONE] {message}")
""",
)

# ── data/ ────────────────────────────────────────────────────────────────────
_write("data/__init__.py", "")
_write(
    "data/email_data.py",
    """
from dataclasses import dataclass
import numpy as np

SENDER_POOLS = {
    "high":   ["professor@college.edu", "hod@college.edu",     "director@company.com"],
    "medium": ["faculty@college.edu",   "teammate@project.com", "hr@company.com"],
    "low":    ["news@techsite.com",      "promo@shopping.com",   "noreply@newsletter.com"],
}
SUBJECT_POOLS = {
    "high":   ["URGENT: Meeting in 1 hour",     "Action required: deadline today", "Critical issue needs attention"],
    "medium": ["Assignment submission reminder", "Team meeting tomorrow",           "Project status update needed"],
    "low":    ["Weekly tech newsletter",         "New offers just for you",         "Monthly digest: top stories"],
}
PRIORITY_MAP = {"high": 3, "medium": 2, "low": 1}

@dataclass
class Email:
    subject:            str
    sender:             str
    priority:           int
    sender_importance:  int
    waiting_time:       int
    workload:           int
    time_of_day:        int

    def to_state_vector(self):
        import numpy as np
        return np.array([
            self.priority, self.sender_importance,
            self.waiting_time, self.workload, self.time_of_day,
        ], dtype=np.float32)
""",
)
_write(
    "data/enron_loader.py",
    """
import os, re, random
from dataclasses import dataclass

@dataclass
class RawEmail:
    subject: str
    sender:  str

class EnronLoader:
    _SUBJECT_RE = re.compile(r"^Subject:\\s*(.+)$", re.MULTILINE | re.IGNORECASE)
    _FROM_RE    = re.compile(r"^From:\\s*(.+)$",    re.MULTILINE | re.IGNORECASE)

    def __init__(self, dataset_path, max_emails=50_000):
        self.dataset_path = dataset_path
        self.max_emails   = max_emails

    def load(self):
        emails = []
        for root, _, files in os.walk(self.dataset_path):
            for fname in files:
                if fname.startswith("."): continue
                raw = self._parse(os.path.join(root, fname))
                if raw: emails.append(raw)
                if len(emails) >= self.max_emails: break
            if len(emails) >= self.max_emails: break
        random.shuffle(emails)
        print(f"  Enron: loaded {len(emails):,} emails from {self.dataset_path}")
        return emails

    def _parse(self, path):
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()
            sm = self._SUBJECT_RE.search(text)
            fm = self._FROM_RE.search(text)
            if not sm or not fm: return None
            subject = sm.group(1).strip()
            sender  = fm.group(1).strip()
            if not subject or subject.lower() in ("", "fw:", "re:"): return None
            if len(subject) < 3: return None
            return RawEmail(subject=subject, sender=sender)
        except Exception:
            return None
""",
)

# ── agent/ ───────────────────────────────────────────────────────────────────
_write("agent/__init__.py", "")
_write(
    "agent/base.py",
    """
from abc import ABC, abstractmethod
import numpy as np

class BaseAgent(ABC):
    @abstractmethod
    def select_action(self, state): pass
    @abstractmethod
    def learn(self, state, action, reward, next_state, done): pass
""",
)
_write(
    "agent/dqn.py",
    """
import numpy as np, random
from collections import deque
import torch, torch.nn as nn, torch.optim as optim
from agent.base import BaseAgent

class QNetwork(nn.Module):
    def __init__(self, state_size=5, action_size=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 128), nn.ReLU(),
            nn.Linear(128, 128),        nn.ReLU(),
            nn.Linear(128, action_size),
        )
    def forward(self, x): return self.net(x)

class ReplayBuffer:
    def __init__(self, capacity=100_000):
        self._buf = deque(maxlen=capacity)
    def push(self, s, a, r, ns, d): self._buf.append((s,a,r,ns,d))
    def sample(self, bs):
        b = random.sample(self._buf, bs)
        s,a,r,ns,d = zip(*b)
        return (torch.tensor(np.array(s),  dtype=torch.float32),
                torch.tensor(a,            dtype=torch.long),
                torch.tensor(r,            dtype=torch.float32),
                torch.tensor(np.array(ns), dtype=torch.float32),
                torch.tensor(d,            dtype=torch.float32))
    def __len__(self): return len(self._buf)

class DQNAgent(BaseAgent):
    def __init__(self, state_size=5, action_size=4, alpha=0.0005, gamma=0.95,
                 epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.9999995,
                 batch_size=128, target_update=1000, buffer_capacity=100_000):
        self.action_size   = action_size
        self.gamma         = gamma
        self.epsilon       = epsilon
        self.epsilon_min   = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.batch_size    = batch_size
        self.target_update = target_update
        self._episode      = 0
        self._device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"  DQN device: {self._device}")
        self._policy_net = QNetwork(state_size, action_size).to(self._device)
        self._target_net = QNetwork(state_size, action_size).to(self._device)
        self._target_net.load_state_dict(self._policy_net.state_dict())
        self._target_net.eval()
        self._optimizer = optim.Adam(self._policy_net.parameters(), lr=alpha)
        self._buffer    = ReplayBuffer(buffer_capacity)

    def select_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.action_size)
        with torch.no_grad():
            t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self._device)
            return int(self._policy_net(t).argmax().item())

    def learn(self, state, action, reward, next_state, done):
        self._buffer.push(state, action, reward, next_state, done)
        if len(self._buffer) < self.batch_size: return
        s,a,r,ns,d = [x.to(self._device) for x in self._buffer.sample(self.batch_size)]
        cur_q = self._policy_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            tgt_q = r + self.gamma * self._target_net(ns).max(1).values * (1-d)
        loss = nn.MSELoss()(cur_q, tgt_q)
        self._optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self._policy_net.parameters(), 1.0)
        self._optimizer.step()

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        self._episode += 1
        if self._episode % self.target_update == 0:
            self._target_net.load_state_dict(self._policy_net.state_dict())

    @property
    def q_table_size(self): return len(self._buffer)
""",
)
_write(
    "agent/q_learning.py",
    """
import numpy as np, pickle
from collections import defaultdict
from agent.base import BaseAgent

class QLearningAgent(BaseAgent):
    def __init__(self, state_size=5, action_size=4, alpha=0.1, gamma=0.95,
                 epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.995):
        self.action_size   = action_size
        self.alpha         = alpha
        self.gamma         = gamma
        self.epsilon       = epsilon
        self.epsilon_min   = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.q_table       = defaultdict(lambda: np.zeros(action_size))

    def select_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.action_size)
        return int(np.argmax(self.q_table[self._key(state)]))

    def learn(self, state, action, reward, next_state, done):
        s, s_ = self._key(state), self._key(next_state)
        future = 0.0 if done else np.max(self.q_table[s_])
        self.q_table[s][action] += self.alpha * ((reward + self.gamma*future) - self.q_table[s][action])

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def save(self, path):
        with open(path, "wb") as f: pickle.dump(dict(self.q_table), f)
        print(f"Model saved -> {path}")

    def load(self, path):
        with open(path, "rb") as f: data = pickle.load(f)
        self.q_table = defaultdict(lambda: np.zeros(self.action_size), data)
        self.epsilon = self.epsilon_min

    @staticmethod
    def _key(state): return tuple(state.astype(int).tolist())

    @property
    def q_table_size(self): return len(self.q_table)
""",
)

# ── environment/ ─────────────────────────────────────────────────────────────
_write("environment/__init__.py", "")
_write(
    "environment/base.py",
    """
from abc import ABC, abstractmethod
import numpy as np

class BaseEnvironment(ABC):
    @abstractmethod
    def reset(self): pass
    @abstractmethod
    def step(self, action): pass
""",
)
_write(
    "environment/reward.py",
    """
class RewardCalculator:
    REPLY_NOW=0; DELAY_REPLY=1; MARK_IMPORTANT=2; ARCHIVE=3
    def calculate(self, email, action):
        if email.priority == 3: return {0:+10.0,1:-5.0,2:+3.0,3:-8.0}[action]
        if email.priority == 2: return {0:-3.0,1:+2.0,2:+6.0,3:-2.0}[action]
        return {0:-3.0,1:+1.0,2:-1.0,3:+5.0}[action]
""",
)
_write(
    "environment/email_env.py",
    """
import numpy as np
from environment.base import BaseEnvironment
from environment.reward import RewardCalculator
from simulation.simulator import EmailSimulator

class EmailEnvironment(BaseEnvironment):
    STATE_SIZE=5; ACTION_SIZE=4
    def __init__(self, simulator, max_steps=50):
        self._simulator   = simulator
        self._reward_calc = RewardCalculator()
        self._max_steps   = max_steps
        self._current_email = None
        self._step_count = 0
    def reset(self):
        self._step_count = 0
        self._current_email = self._simulator.next_email()
        return self._current_email.to_state_vector()
    def step(self, action):
        reward = self._reward_calc.calculate(self._current_email, action)
        self._step_count += 1
        done = self._step_count >= self._max_steps
        self._current_email = self._simulator.next_email()
        return self._current_email.to_state_vector(), reward, done
    @property
    def current_email(self): return self._current_email
    @property
    def action_labels(self): return {0:"reply_now",1:"delay_reply",2:"mark_important",3:"archive"}
""",
)

# ── simulation/ ──────────────────────────────────────────────────────────────
_write("simulation/__init__.py", "")
_write(
    "simulation/simulator.py",
    """
import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from data.email_data import Email
from simulation.sources.base import EmailSource

class EmailSimulator:
    def __init__(self, source): self._source = source
    def next_email(self): return self._source.get_email()
""",
)
_write("simulation/sources/__init__.py", "")
_write(
    "simulation/sources/base.py",
    """
from abc import ABC, abstractmethod
from data.email_data import Email

class EmailSource(ABC):
    @abstractmethod
    def get_email(self): pass
""",
)
_write(
    "simulation/sources/nlp_extractor.py",
    """
import re
from datetime import datetime
from data.email_data import Email

class NLPEmailExtractor:
    _HIGH   = ["urgent","asap","immediately","critical","emergency","today","deadline",
               "now","action required","due today","final","overdue","meeting in",
               "last chance","important","respond","response needed","time sensitive","expiring"]
    _MED    = ["reminder","follow up","update","review","schedule","tomorrow","next week",
               "please","request","submission","assignment","interview","project","team",
               "discuss","feedback","confirm","invitation","rescheduled"]
    _LOW    = ["newsletter","unsubscribe","promo","offer","deal","discount","sale","digest",
               "weekly","monthly","notification","no-reply","noreply","update available",
               "new arrivals","coupon","subscribe","marketing"]
    _HDOM   = [".edu",".ac.in",".ac.uk",".gov",".ac.","university","college","institute",
               "school","hospital","police"]
    _LPAT   = ["noreply","no-reply","newsletter","promo","notification","alert","mailer",
               "donotreply","marketing","digest","deals","offers","unsubscribe","support@","info@"]

    def extract(self, subject, sender):
        priority = self._priority(subject)
        simp     = self._sender_imp(sender)
        workload = self._workload()
        return Email(subject=subject, sender=sender, priority=priority,
                     sender_importance=simp, waiting_time=0,
                     workload=workload, time_of_day=datetime.now().hour)

    def explain(self, subject, sender):
        p  = self._priority(subject); si = self._sender_imp(sender)
        wl = self._workload(); h = datetime.now().hour
        return {"priority":p,"sender_importance":si,"waiting_time":0,
                "workload":wl,"time_of_day":h,
                "priority_reason":{3:"high-urgency keyword",2:"moderate keyword",1:"no urgency"}[p],
                "sender_reason":{3:"academic/gov domain",2:"professional",1:"newsletter/promo"}[si],
                "workload_reason":{1:f"off-hours ({h}:00)",2:f"work hours ({h}:00)",3:f"peak ({h}:00)"}[wl]}

    def _priority(self, subject):
        t = subject.lower()
        if any(k in t for k in self._HIGH): return 3
        if any(k in t for k in self._MED):  return 2
        if any(k in t for k in self._LOW):  return 1
        return 2

    def _sender_imp(self, sender):
        s = sender.lower()
        if any(p in s for p in self._LPAT): return 1
        if any(d in s for d in self._HDOM): return 3
        return 2

    def _workload(self):
        h = datetime.now().hour
        if 9<=h<=11 or 14<=h<=16: return 3
        if 8<=h<=18: return 2
        return 1
""",
)
_write(
    "simulation/sources/enron.py",
    """
import random
from data.email_data import Email
from data.enron_loader import EnronLoader, RawEmail
from simulation.sources.base import EmailSource
from simulation.sources.nlp_extractor import NLPEmailExtractor

class EnronEmailSource(EmailSource):
    def __init__(self, dataset_path, max_emails=50_000):
        loader          = EnronLoader(dataset_path, max_emails)
        self._emails    = loader.load()
        self._extractor = NLPEmailExtractor()
        self._index     = 0
        random.shuffle(self._emails)
    def get_email(self):
        raw = self._emails[self._index % len(self._emails)]
        self._index += 1
        return self._extractor.extract(raw.subject, raw.sender)
    def __len__(self): return len(self._emails)
""",
)

# ── training/ ────────────────────────────────────────────────────────────────
_write("training/__init__.py", "")
_write(
    "training/trainer.py",
    """
import pickle, os, time
from utils.logger import Logger

class Trainer:
    def __init__(self, env, agent, episodes=1000, log_every=100,
                 checkpoint_every=0, checkpoint_dir="models"):
        self._env              = env
        self._agent            = agent
        self._episodes         = episodes
        self._logger           = Logger(log_every)
        self._checkpoint_every = checkpoint_every
        self._checkpoint_dir   = checkpoint_dir

    def run(self):
        self._logger.section(f"Training Started — {self._episodes:,} episodes")
        history = []; start = time.time()
        for ep in range(1, self._episodes+1):
            reward = self._run_episode()
            history.append(reward)
            self._agent.decay_epsilon()
            self._logger.episode(ep, reward, self._agent.epsilon, self._agent.q_table_size)
            if self._checkpoint_every and ep % self._checkpoint_every == 0:
                path = os.path.join(self._checkpoint_dir, f"checkpoint_ep{ep}.pkl")
                self._save_agent(path)
                print(f"  {(time.time()-start)/60:.1f} min | checkpoint -> {path}")
        self._logger.done(f"Training complete — {(time.time()-start)/60:.1f} min")
        return history

    def _run_episode(self):
        state = self._env.reset(); total = 0.0; done = False
        while not done:
            action = self._agent.select_action(state)
            next_state, reward, done = self._env.step(action)
            self._agent.learn(state, action, reward, next_state, done)
            state = next_state; total += reward
        return total

    def save(self, path):
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        self._save_agent(path)

    def _save_agent(self, path):
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        with open(path, "wb") as f: pickle.dump(self._agent, f)
        print(f"  Saved -> {path}")

    @staticmethod
    def load(path):
        with open(path, "rb") as f: return pickle.load(f)
""",
)
_write(
    "training/evaluator.py",
    """
import numpy as np
import matplotlib.pyplot as plt

class Evaluator:
    OPTIMAL_ACTIONS = {3:0, 2:2, 1:3}

    def __init__(self, env, agent):
        self._env = env; self._agent = agent

    def evaluate(self, episodes=100):
        saved = self._agent.epsilon; self._agent.epsilon = 0.0
        rewards, correct, total = [], 0, 0
        for _ in range(episodes):
            state = self._env.reset(); ep_r = 0.0; done = False
            while not done:
                action = self._agent.select_action(state)
                next_state, reward, done = self._env.step(action)
                correct += int(action == self.OPTIMAL_ACTIONS[int(state[0])])
                total += 1; ep_r += reward; state = next_state
            rewards.append(ep_r)
        self._agent.epsilon = saved
        return {"mean_reward":round(float(np.mean(rewards)),2),
                "accuracy":round(correct/total*100,1),
                "min_reward":round(float(np.min(rewards)),2),
                "max_reward":round(float(np.max(rewards)),2)}

    def print_results(self, results):
        print(f"  Mean reward : {results[\'mean_reward\']:+.2f}")
        print(f"  Accuracy    : {results[\'accuracy\']}%")

    def plot_rewards(self, history, window=50, save_path="reward_curve.png"):
        episodes = range(1, len(history)+1)
        rolling  = np.convolve(history, np.ones(window)/window, mode="valid")
        fig, ax  = plt.subplots(figsize=(10,5))
        ax.plot(episodes, history, color="#94a3b8", alpha=0.4, linewidth=0.8, label="Raw")
        ax.plot(episodes[window-1:], rolling, color="#38bdf8", linewidth=2.2, label=f"Rolling avg")
        ax.set_xlabel("Episode"); ax.set_ylabel("Total Reward")
        ax.legend(); ax.grid(True, alpha=0.2); fig.tight_layout()
        fig.savefig(save_path, dpi=150); plt.close(fig)
        print(f"  Plot saved -> {save_path}")
""",
)

print("All source files written to disk!")
import importlib, sys

# Flush any previous module cache so fresh versions are used
for _mod in list(sys.modules.keys()):
    if any(
        _mod.startswith(x)
        for x in ["config", "data", "agent", "environment", "simulation", "training", "utils"]
    ):
        del sys.modules[_mod]
print("Module cache cleared. Ready to import.")

In [ ]:
# Cell 2: GPU Check
import torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
else:
    print("No GPU — training will use CPU (slower)")

In [ ]:
# Cell 3: Download Enron dataset into enron_dataset/ folder
import os

try:
    _ = ENRON_DIR
except NameError:
    try:
        _ = PROJECT_DIR
    except NameError:
        PROJECT_DIR = os.path.abspath("")
    ENRON_DIR = os.path.join(PROJECT_DIR, "enron_dataset")

os.makedirs(ENRON_DIR, exist_ok=True)
print(f"Dataset will be stored in: {ENRON_DIR}")

email_count = sum(len(files) for _, _, files in os.walk(ENRON_DIR))
if email_count > 0:
    print(f"Enron dataset already present: {email_count:,} files")
else:
    TAR_PATH = os.path.join(PROJECT_DIR, "enron.tar.gz")
    URL = "https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz"
    print("Downloading 1.7 GB dataset from CMU...")
    ret = os.system(f'wget -q --show-progress -O "{TAR_PATH}" "{URL}"')
    if ret != 0:
        raise RuntimeError("wget download failed.")
    print("Extracting into enron_dataset/ ...")
    ret = os.system(f'tar -xzf "{TAR_PATH}" -C "{ENRON_DIR}" --strip-components=1')
    if ret != 0:
        raise RuntimeError("tar extraction failed.")
    os.remove(TAR_PATH)
    email_count = sum(len(files) for _, _, files in os.walk(ENRON_DIR))
    print(f"Done! {email_count:,} email files stored in: {ENRON_DIR}")

In [ ]:
# Cell 4: Verify all imports + sample emails
import os, sys

_cwd = os.path.abspath("")
if "email_response_timing" in _cwd:
    PROJECT_DIR = _cwd[: _cwd.find("email_response_timing") + len("email_response_timing(REL)")]
else:
    PROJECT_DIR = _cwd
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

try:
    _ = ENRON_DIR
except NameError:
    ENRON_DIR = os.path.join(PROJECT_DIR, "enron_dataset")
try:
    _ = MODELS_DIR
except NameError:
    MODELS_DIR = os.path.join(PROJECT_DIR, "models")
try:
    _ = CKPT_DIR
except NameError:
    CKPT_DIR = os.path.join(MODELS_DIR, "checkpoints")
try:
    _ = DQN_PATH
except NameError:
    DQN_PATH = os.path.join(MODELS_DIR, "dqn.pkl")
try:
    _ = QL_PATH
except NameError:
    QL_PATH = os.path.join(MODELS_DIR, "qlearning.pkl")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

from config import Config
from data.enron_loader import EnronLoader
from simulation.sources.enron import EnronEmailSource
from simulation.simulator import EmailSimulator
from environment.email_env import EmailEnvironment
from agent.dqn import DQNAgent
from agent.q_learning import QLearningAgent
from training.trainer import Trainer
from training.evaluator import Evaluator

print("All imports OK")
print("Sample Enron emails:")
loader = EnronLoader(ENRON_DIR, max_emails=6)
for e in loader.load():
    print(f"  Subject : {e.subject[:60]}")
    print(f"  Sender  : {e.sender}")
    print()

In [ ]:
# Cell 5: Train DQN — 100,000 episodes (GPU)
import os, sys

_cwd = os.path.abspath("")
PROJECT_DIR = (
    _cwd[: _cwd.find("email_response_timing") + len("email_response_timing(REL)")]
    if "email_response_timing" in _cwd
    else _cwd
)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

try:
    _ = ENRON_DIR
except NameError:
    ENRON_DIR = os.path.join(PROJECT_DIR, "enron_dataset")
try:
    _ = MODELS_DIR
except NameError:
    MODELS_DIR = os.path.join(PROJECT_DIR, "models")
try:
    _ = CKPT_DIR
except NameError:
    CKPT_DIR = os.path.join(MODELS_DIR, "checkpoints")
try:
    _ = DQN_PATH
except NameError:
    DQN_PATH = os.path.join(MODELS_DIR, "dqn.pkl")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

from config import Config
from simulation.sources.enron import EnronEmailSource
from simulation.simulator import EmailSimulator
from environment.email_env import EmailEnvironment
from agent.dqn import DQNAgent
from training.trainer import Trainer

EPISODES = 100_000
enron_src = EnronEmailSource(ENRON_DIR, max_emails=50_000)
sim = EmailSimulator(enron_src)
env = EmailEnvironment(sim, max_steps=Config.MAX_STEPS)
dqn = DQNAgent(
    alpha=Config.DQN_ALPHA,
    gamma=Config.DQN_GAMMA,
    epsilon=Config.DQN_EPSILON,
    epsilon_min=Config.DQN_EPSILON_MIN,
    epsilon_decay=Config.DQN_EPSILON_DECAY,
    batch_size=Config.DQN_BATCH_SIZE,
    target_update=Config.DQN_TARGET_UPDATE,
    buffer_capacity=Config.DQN_BUFFER_CAP,
)
trainer_dqn = Trainer(
    env=env,
    agent=dqn,
    episodes=EPISODES,
    log_every=10_000,
    checkpoint_every=25_000,
    checkpoint_dir=CKPT_DIR,
)
dqn_history = trainer_dqn.run()
trainer_dqn.save(DQN_PATH)
print(f"DQN saved to {DQN_PATH}")

In [ ]:
# Cell 6: Train Q-Learning — 100,000 episodes
import os, sys

_cwd = os.path.abspath("")
PROJECT_DIR = (
    _cwd[: _cwd.find("email_response_timing") + len("email_response_timing(REL)")]
    if "email_response_timing" in _cwd
    else _cwd
)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

try:
    _ = ENRON_DIR
except NameError:
    ENRON_DIR = os.path.join(PROJECT_DIR, "enron_dataset")
try:
    _ = MODELS_DIR
except NameError:
    MODELS_DIR = os.path.join(PROJECT_DIR, "models")
try:
    _ = CKPT_DIR
except NameError:
    CKPT_DIR = os.path.join(MODELS_DIR, "checkpoints")
try:
    _ = QL_PATH
except NameError:
    QL_PATH = os.path.join(MODELS_DIR, "qlearning.pkl")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

from config import Config
from simulation.sources.enron import EnronEmailSource
from simulation.simulator import EmailSimulator
from environment.email_env import EmailEnvironment
from agent.q_learning import QLearningAgent
from training.trainer import Trainer

EPISODES = 100_000
enron_src_ql = EnronEmailSource(ENRON_DIR, max_emails=50_000)
sim_ql = EmailSimulator(enron_src_ql)
env_ql = EmailEnvironment(sim_ql, max_steps=Config.MAX_STEPS)
ql = QLearningAgent(
    alpha=Config.QL_ALPHA,
    gamma=Config.QL_GAMMA,
    epsilon=Config.QL_EPSILON,
    epsilon_min=Config.QL_EPSILON_MIN,
    epsilon_decay=Config.QL_EPSILON_DECAY,
)
trainer_ql = Trainer(
    env=env_ql,
    agent=ql,
    episodes=EPISODES,
    log_every=10_000,
    checkpoint_every=25_000,
    checkpoint_dir=CKPT_DIR,
)
ql_history = trainer_ql.run()
ql.save(QL_PATH)
print(f"Q-Learning saved to {QL_PATH}")

In [ ]:
# Cell 7: Complete — download trained models
import os

try:
    _dqn = DQN_PATH
except NameError:
    _dqn = os.path.join(os.path.abspath(""), "models", "dqn.pkl")
try:
    _ql = QL_PATH
except NameError:
    _ql = os.path.join(os.path.abspath(""), "models", "qlearning.pkl")

print("=" * 55)
print("  TRAINING COMPLETE")
print(f"  DQN model  : {_dqn}")
print(f"  QL model   : {_ql}")
print("=" * 55)

# Download models to your local machine
try:
    from google.colab import files

    if os.path.exists(_dqn):
        files.download(_dqn)
    if os.path.exists(_ql):
        files.download(_ql)
    print("Models downloaded! Place them in your local models/ folder.")
except ImportError:
    print("Running locally — models saved to models/ folder.")